# Fine-tune Qwen3.5-4B on RBI Regulatory QA Dataset (QLoRA)

**Dataset**: `Magneto/qa_with_personas` — 23,892 Q&A pairs generated from RBI (Reserve Bank of India) circulars and master directions. Despite the dataset card saying "BI", this is RBI data (find-and-replace typo in the original README).

**Format**: RAG-style — `question + context -> answer`. The model learns to answer using a provided regulatory passage, which is the right setup since you'll eventually pair this with a retriever (or just always supply the relevant context at inference time).

**Target model**: `Qwen/Qwen3.5-4B`, QLoRA (4-bit NF4 + LoRA adapters).

## This is NOT just a model-name swap from the Qwen3-4B notebook

Qwen3.5 is a genuinely different, newer architecture from Qwen3 -- not a version bump of the same
design. The differences that actually affect this notebook's code:

- **Hybrid attention, not pure attention.** Qwen3.5-4B uses a 3:1 stack of Gated DeltaNet
  (linear attention) layers to Gated Attention (full attention) layers -- 24 DeltaNet layers,
  8 full-attention layers, 32 total. Only the 8 full-attention layers use the familiar
  `q_proj`/`k_proj`/`v_proj`/`o_proj` naming that LoRA targets by name; the DeltaNet layers have
  a different internal structure. This notebook targets the named attention projections only
  (same approach as the Qwen3 notebook, and the one with the most precedent in published
  fine-tuning work on this exact model) -- see the LoRA config cell for the `all-linear`
  alternative and its tradeoffs.
- **Natively multimodal checkpoint, text-only usage.** The `Qwen/Qwen3.5-4B` checkpoint on the
  Hub includes vision-tower weights (it's an image+video+text model). We load it with
  `AutoModelForCausalLM`, which resolves to the text-only `Qwen3_5ForCausalLM` class and skips
  the vision weights entirely -- we never touch images here, this is a text-only RAG QA task.
- **Larger vocabulary.** ~248,320 tokens vs Qwen3's ~151,936 -- about 1.6x bigger embedding
  table and LM head. This adds some fixed memory/compute cost independent of model size that
  the runtime-budget estimate below accounts for.
- **Thinking is on by default**, and disabled the same way as Qwen3 (`enable_thinking=False` in
  `apply_chat_template`) -- this part of the pipeline is unchanged.
- **Optional fast kernels** (`causal_conv1d`, `fla`) accelerate the DeltaNet layers but aren't
  installed by default on Colab. Without them the model still works, just slower -- this
  notebook budgets for the slower, no-extra-kernels path since installing custom CUDA kernel
  packages reliably on a shared Colab T4 isn't something to bet a fixed time budget on.

**Runtime budget**: targets **under 5.5 hours on a Colab T4**, same ceiling as the Qwen3-4B and
Qwen3-1.7B notebooks it's paired with. Because of the architectural uncertainty above, the
per-step time estimate here is less precise than for the same-family Qwen3-1.7B notebook -- so
this notebook uses a SMALLER data budget and a BIGGER safety margin, and leans more heavily on
the live runtime-probe cell (added before the real training call) to correct course with real
measured numbers rather than the upfront estimate alone.

Run cells top to bottom. Cell with model/dataset config is the one you'll touch most; the
GPU-detection cell right before it sets sensible defaults automatically.


## 1. Install dependencies

`Qwen3_5ForCausalLM` is in mainline `transformers` -- support landed in **v5.2.0** (confirmed by
the model card and multiple downstream framework changelogs), so this is a real PyPI release
pin, NOT a from-source/bleeding-edge-main install. If your environment somehow resolves an
older `transformers` despite the pin below (e.g. a stale Colab base image), the version-check
cell right after this one will fail loudly rather than silently giving you a confusing
`model type qwen3_5 not recognized` error later. If you're on Colab, run this once and restart
the runtime if prompted.


In [1]:
!pip install -q -U "transformers>=5.2.0" accelerate peft bitsandbytes trl datasets evaluate sentencepiece tensorboard matplotlib wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.2 MB/s eta 0:00:00


In [2]:
import torch, transformers, peft, trl, bitsandbytes
from packaging import version

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# Qwen3.5 needs transformers >= 5.2.0 (the version that added Qwen3_5ForCausalLM). Failing here
# with a clear message beats failing three cells later at model load with a cryptic
# "model type qwen3_5 not recognized" error.
assert version.parse(transformers.__version__) >= version.parse("5.2.0"), (
    f"transformers {transformers.__version__} is too old for Qwen3.5 (needs >= 5.2.0). "
    f"Re-run the pip install cell above, then Runtime > Restart runtime, then re-run from the top."
)
print("\ntransformers version OK for Qwen3.5.")

torch: 2.11.0+cu128 | CUDA available: True
transformers: 5.12.1
peft: 0.19.1
trl: 1.7.0
bitsandbytes: 0.49.2
GPU: Tesla T4
VRAM total: 15.6 GB

transformers version OK for Qwen3.5.


### Weights & Biases login

Paste your W&B API key when prompted (find it at https://wandb.ai/authorize). `getpass` hides the
characters as you type and — importantly — does NOT save the key anywhere in this notebook file, so it's
safe to share/upload the `.ipynb` afterward without leaking your key. You only need to do this once per
Colab session; it'll stay logged in until the runtime restarts.

In [3]:
import wandb
from getpass import getpass

wandb_api_key = getpass("Paste your W&B API key (input hidden): ")
wandb.login(key=wandb_api_key)

# Clear the key from this variable immediately after use — it's no longer needed,
# and there's no reason to keep a copy of the secret sitting in a live notebook variable.
del wandb_api_key
print("W&B login successful.")

Paste your W&B API key (input hidden): ··········


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nidheshgomai2006 (nidheshgomai2006-k-j-somaiya-school-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful.


### Detect GPU and set platform-appropriate defaults

**Correction from an earlier version of this notebook**: we previously set `USE_BF16=False` (i.e. train in
fp16) on T4 for a speed win, since T4 lacks fast native bf16 tensor cores. That was wrong for this specific
model family — Qwen3's native parameters/buffers are bf16, and mixing them with PyTorch's fp16 `GradScaler`
crashes outright (`NotImplementedError: ..._unscale_cuda not implemented for 'BFloat16'`), because the scaler
only knows how to handle fp16 gradients. This isn't a tunable setting for Qwen3 — bf16 training is required
regardless of GPU generation. T4 will be slower in bf16 than it would be in fp16 on an fp16-native model, but
that's a speed cost, not a correctness option, here.

We still use GPU detection to decide gradient checkpointing / micro-batch size (those are still legitimately
GPU-memory-dependent), just no longer to pick a training dtype.

In [4]:
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
_is_t4 = "T4" in _gpu_name
_is_old_card = _is_t4 or "P100" in _gpu_name or "K80" in _gpu_name

# Always bf16: Qwen3's native dtype is bf16, and fp16 + GradScaler is incompatible with it
# (crashes in torch.amp.grad_scaler regardless of GPU generation). Not a tunable choice here.
USE_BF16 = True

if _is_old_card:
    # T4/P100/K80: keep grad checkpointing on as a memory safety margin. The SFT chunked_nll
    # loss path is memory-safe on its own, but TRL also computes a per-token entropy metric for
    # logging that — on a non-chunked fallback path — can materialize a full
    # [batch*seq_len, vocab_size] tensor (~150k vocab for Qwen3). Checkpointing buys headroom
    # against that even though the primary loss path doesn't need it.
    USE_GRAD_CHECKPOINTING = True
    SUGGESTED_MICRO_BATCH = 1
else:
    USE_GRAD_CHECKPOINTING = True   # keep on for smaller/constrained GPUs (e.g. your local 3050)
    SUGGESTED_MICRO_BATCH = 1

print(f"Detected GPU: {_gpu_name or 'none'}")
print(f"USE_BF16={USE_BF16}  USE_GRAD_CHECKPOINTING={USE_GRAD_CHECKPOINTING}  "
      f"SUGGESTED_MICRO_BATCH={SUGGESTED_MICRO_BATCH}")
print("These feed into the config cell below — override there if you know better for your setup.")

Detected GPU: Tesla T4
USE_BF16=True  USE_GRAD_CHECKPOINTING=True  SUGGESTED_MICRO_BATCH=1
These feed into the config cell below — override there if you know better for your setup.


### Why these specific numbers? (runtime-budget reasoning, with the uncertainty flagged honestly)

This notebook targets **Qwen3.5-4B** against the same **5.5-hour Colab T4** ceiling as its
siblings. The estimate here is **less precise** than the Qwen3-1.7B notebook's, and the config
below is deliberately more conservative as a result -- here's why.

**The anchor point**: a real timed run of the Qwen3-4B notebook (attention-only LoRA,
`MAX_SEQ_LENGTH=384`, micro-batch 1, effective batch 16) measured **76.8 seconds/step** on a T4.

**Why Qwen3.5-4B can't just reuse that number directly:**
- It's a similar total parameter count (~4B) but a genuinely different architecture -- a 3:1
  Gated DeltaNet (linear attention) / Gated Attention hybrid, not pure full attention. Linear
  attention layers have a different compute profile than full attention; whether that nets out
  faster or slower per-step than equivalent full-attention layers depends on implementation
  details we can't verify without actually running it.
- ~1.6x larger vocabulary (248,320 vs ~151,936) means a bigger embedding table and LM head,
  which adds memory and compute cost independent of the hyperparameters above.
- The DeltaNet layers have optional fast CUDA kernels (`causal_conv1d`, `fla`) that are NOT
  installed by default on Colab. Without them, those layers fall back to slower, more
  memory-hungry plain PyTorch ops -- and we're budgeting for that slower path, since reliably
  installing custom CUDA kernel packages on shared Colab infrastructure isn't something to bet
  a fixed time budget on.

**The estimate**: treating Qwen3.5-4B as roughly comparable in scale to Qwen3-4B but applying a
1.25x time penalty for the three factors above gives **~96 sec/step** -- clearly labeled as an
estimate with real uncertainty, not a measurement.

**The budget**: at ~96 sec/step, with ~12 minutes of fixed overhead (installs, larger
tokenizer/vocab downloads) and a **25% safety margin** (bigger than the 1.7B notebook's 15%,
specifically because this estimate is shakier), the 5.5h ceiling allows about **150 training
steps**. With effective batch 16 and `NUM_EPOCHS=1`, that's `TRAIN_SUBSET_SIZE=2400` -- fewer
unique examples than either Qwen3 sibling notebook, and only 1 epoch instead of 2, because a
single conservative pass over more unique data is a safer bet than multiple epochs when the
per-step timing itself is uncertain.

**This is exactly why the live runtime-probe cell later in this notebook matters more here than
in the other two notebooks.** It measures your *actual* seconds/step on the *actual* assigned
GPU with the *actual* installed kernels (fast or fallback) before you commit to the long run --
treat its output as the real source of truth, and the numbers above as just the starting plan.


## 2. Config

Set `MODEL_NAME` to try 4B first. If you hit `CUDA out of memory` during training, just change this one line to the 1.7B fallback and re-run from here — no other code changes needed.

**Sized down for a fast smoke test**: `TRAIN_SUBSET_SIZE=200`, 1 epoch, `MAX_SEQ_LENGTH=384` — together this drops total steps from 250 to roughly 13, so you can confirm the whole pipeline (load, LoRA, train, save, generate) actually completes within a constrained Colab session before committing to a longer run. Once this finishes cleanly, bump `TRAIN_SUBSET_SIZE` up (or to `None` for the full 19,113-example train set), restore `NUM_EPOCHS=2`, and re-check `MAX_SEQ_LENGTH` against Cell 3a's real percentiles.

In [29]:
# ============================================================
# MODEL SELECTION
# ============================================================
# This notebook is dedicated to Qwen3.5-4B specifically -- no fallback to a different model.
# The Hub checkpoint is the full multimodal release (it includes vision-tower weights for
# image/video input), but AutoModelForCausalLM resolves "qwen3_5" to the text-only
# Qwen3_5ForCausalLM class, which explicitly skips loading vision weights
# (_keys_to_ignore_on_load_unexpected includes "model.visual.*") -- so we get a clean text-only
# load from the same checkpoint, no separate text-only repo needed.
MODEL_NAME = "Qwen/Qwen3.5-4B"  # ~4 billion parameters (dense, hybrid linear/full attention)

# --- Dataset ---
DATASET_NAME = "Magneto/qa_with_personas"  # the RBI regulatory Q&A dataset on Hugging Face Hub

# ============================================================
# SEQUENCE LENGTH
# ============================================================
# MAX_SEQ_LENGTH = the maximum number of TOKENS the model will see per training example, after
# combining the system prompt + context + question + answer. Qwen3.5 uses a DIFFERENT tokenizer
# than Qwen3 (much larger vocab -- 248,320 vs ~151,936), so the token-length percentiles
# measured in Cell 3a for the Qwen3 notebooks do NOT carry over unchanged here. Cell 3a below
# re-measures the distribution using Qwen3.5's own tokenizer -- run it before trusting this
# number for anything beyond the value already set here. A larger vocabulary generally means
# MORE efficient tokenization (each token can represent more text on average), so if anything
# we'd expect Qwen3.5 to need a slightly SHORTER MAX_SEQ_LENGTH than Qwen3 for the same text --
# 384 is kept as the starting point, but treat Cell 3a's real output as the authority.
MAX_SEQ_LENGTH = 512

# ============================================================
# LoRA CONFIG (the "adapter" — what actually gets trained)
# ============================================================
# LoRA = Low-Rank Adaptation. Instead of updating all ~4 billion parameters of the base model
# (which would need far more VRAM than we have), we freeze the base model entirely and insert
# small trainable "adapter" matrices into specific layers. Only those small matrices get updated
# during training — typically well under 1% of the total parameter count.

# LORA_R ("rank"): controls the SIZE of each adapter matrix, and therefore how much new
# information the adapter can encode. Rank 8 is a small, common starting point.
LORA_R = 8

# LORA_ALPHA: a scaling factor applied to the adapter's output before it's added back into the
# frozen base model's computation. Conventionally set to 2x the rank (8 -> 16).
LORA_ALPHA = 16

# LORA_DROPOUT: during training, randomly "turns off" this fraction (5%) of the adapter's
# internal connections on each forward pass, as a regularization technique against overfitting.
LORA_DROPOUT = 0.05

# LORA_TARGET_MODULES: WHICH weight matrices inside each transformer layer get an adapter.
# THIS IS THE PART THAT ACTUALLY CHANGES FOR QWEN3.5'S ARCHITECTURE.
#
# Qwen3.5-4B is a 3:1 hybrid: 24 Gated DeltaNet (linear attention) layers for every 8 Gated
# Attention (full attention) layers, 32 total. Only the 8 full-attention layers use the
# familiar q_proj/k_proj/v_proj/o_proj naming -- a published fine-tuning comparison on this
# exact model (S0 Tuning, arXiv 2604.01168) confirms LoRA on Qwen3.5-4B targeting
# {q,k,v,o}_proj hits "the 8 attention layers" specifically, and reports it as a working,
# precedented baseline configuration. The 24 DeltaNet layers have a different internal
# structure (their own query/key/value-like projections plus a short causal convolution) that
# isn't named q_proj/k_proj/v_proj/o_proj, so the named-list approach below simply does not
# touch them.
#
# We use the SAME named-attention-only approach as the Qwen3 notebooks for consistency and
# because it's the better-evidenced choice. The alternative -- target_modules="all-linear" --
# would also adapt the DeltaNet layers' own linear projections (more of the model gets
# adapted, in principle more capacity), but "all-linear" auto-discovery on a hybrid
# architecture this new is less battle-tested, and some Unsloth/community docs note it as
# "optional" rather than required even for full Qwen3.5 fine-tunes. If you want to experiment
# with it, swap the line below -- just know it's the less-precedented path on this specific
# architecture, and budget extra retry time for layer-name surprises.
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
# LORA_TARGET_MODULES = "all-linear"  # alternative: also adapts the 24 DeltaNet layers -- see note above

# ============================================================
# TRAINING CONFIG
# ============================================================
OUTPUT_DIR = "./qwen3.5-4b-rbi-qa-lora"  # where checkpoints, logs, and the final adapter get saved

# PER_DEVICE_BATCH_SIZE ("micro-batch"): how many training examples are processed together in
# ONE forward+backward pass on the GPU. SUGGESTED_MICRO_BATCH comes from the GPU-detection cell
# above. Note: Qwen3.5-4B's larger vocab/LM-head means somewhat higher activation memory than
# Qwen3-4B at the same micro-batch and MAX_SEQ_LENGTH -- if you OOM even at micro-batch 1, the
# next lever to pull is MAX_SEQ_LENGTH, not this value (it's already at the floor).
PER_DEVICE_BATCH_SIZE = SUGGESTED_MICRO_BATCH

# GRAD_ACCUM_STEPS: kept at the same effective-batch target (16) as the other two notebooks in
# this series, for easy side-by-side comparison.
GRAD_ACCUM_STEPS = 16 // PER_DEVICE_BATCH_SIZE

# NUM_EPOCHS: only 1 here (vs 2 for the Qwen3 notebooks). Given the genuine uncertainty in the
# per-step timing estimate for this architecture (see the runtime-budget cell above), a single
# conservative pass over more unique data is a safer way to spend the time budget than risking
# a second epoch not finishing.
NUM_EPOCHS = 1

# LEARNING_RATE: 2e-4 is a standard LoRA learning rate, unchanged from the Qwen3 notebooks.
LEARNING_RATE = 2e-4

# WARMUP_RATIO: first 3% of steps ramp up linearly, same as the other notebooks.
WARMUP_RATIO = 0.03

# LOGGING_STEPS: how often to print/record a loss value.
LOGGING_STEPS = 10

# SAVE_STEPS / EVAL_STEPS: checkpoint and validation-eval frequency.
SAVE_STEPS = 50
EVAL_STEPS = 50

# ============================================================
# DATA VOLUME -- sized for a ~5.5-hour ceiling on Qwen3.5-4B, with a bigger safety margin
# than the other notebooks in this series (see the runtime-budget markdown cell above)
# ============================================================
# TRAIN_SUBSET_SIZE=2400, NUM_EPOCHS=1, effective batch 16 -> ~150 training steps. At an
# estimated ~96 sec/step (itself a 1.25x penalty over the measured Qwen3-4B baseline, to
# account for the larger vocab and the lack of fast DeltaNet kernels on stock Colab), that's
# ~4.0h of training time -- leaving meaningfully more headroom under the 5.5h ceiling than
# the other two notebooks, specifically because this estimate is the least certain of the
# three. Set TRAIN_SUBSET_SIZE = None only if you are NOT on a fixed Colab session length.
TRAIN_SUBSET_SIZE = 3000
EVAL_SUBSET_SIZE = 200

# ============================================================
# EXPERIMENT TRACKING (Weights & Biases + TensorBoard)
# ============================================================
WANDB_PROJECT = "qwen3-rbi-qa-finetune"

import time as _time
RUN_NAME = (
    f"{MODEL_NAME.split('/')[-1]}"
    f"_subset{TRAIN_SUBSET_SIZE if TRAIN_SUBSET_SIZE else 'FULL'}"
    f"_ep{NUM_EPOCHS}"
    f"_{_time.strftime('%Y%m%d-%H%M%S')}"
)
print("This run will be logged as:", RUN_NAME)


This run will be logged as: Qwen3.5-4B_subset3000_ep1_20260626-080953


## 3. Load and inspect the dataset

By default, `load_dataset` caches downloaded parquet files under `~/.cache/huggingface/datasets/` on the Colab VM's local disk — this is wiped whenever the runtime restarts or disconnects, so you'll re-download (a small, fast download for this dataset) each fresh session. If you'd rather persist the cache across restarts, set `USE_DRIVE_CACHE = True` below to mount Google Drive and point the HF cache there instead. Off by default since it adds a Drive-auth prompt you may not want.

In [6]:
USE_DRIVE_CACHE = False  # set True to persist the HF datasets cache across Colab restarts

if USE_DRIVE_CACHE:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    drive_cache_dir = '/content/drive/MyDrive/hf_datasets_cache'
    os.makedirs(drive_cache_dir, exist_ok=True)
    os.environ['HF_DATASETS_CACHE'] = drive_cache_dir
    print(f"HF datasets cache pointed at: {drive_cache_dir}")
else:
    print("Using default local VM cache (will re-download after a runtime restart).")

Using default local VM cache (will re-download after a runtime restart).


In [7]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print()
print("Example record:")
print(raw_dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.64k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.6M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.66M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4779 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'context', 'document_name', 'persona_role', 'question_type'],
        num_rows: 19113
    })
    validation: Dataset({
        features: ['question', 'answer', 'context', 'document_name', 'persona_role', 'question_type'],
        num_rows: 4779
    })
})

Example record:
{'question': 'As a compliance officer, what is the exact deadline by which all listed Indian companies must submit the specified foreign investment data to the depositories according to the RBI/SEBI circulars?', 'answer': 'The circulars require that all listed Indian companies provide the specified foreign investment data/information to the depositories **before May 15, 2018**. This deadline is explicitly mentioned in the paragraph referring to para 6 of Annexure A of the SEBI circular dated April\u202f05,\u202f2018.', 'context': 'Monitoring of foreign investment limits in listed Indian companies Attention of Authorised Dealer Category-I (AD Ca

### 3a. Check token length distribution before fixing MAX_SEQ_LENGTH

This matters because the dataset card only tells us `context` is ~1000 chars — we should verify actual tokenized length (question + context + answer) before committing to a sequence length, otherwise we either waste VRAM on padding or silently truncate answers. **This re-measurement matters more here than in the Qwen3 notebooks**: Qwen3.5 uses a different tokenizer with a ~1.6x larger vocabulary (248,320 vs ~151,936 tokens), so the token counts for the same text will differ from what was measured for Qwen3 -- don't assume the `MAX_SEQ_LENGTH=384` default carries over unchanged without checking this cell's output.

In [30]:
from transformers import AutoTokenizer
import numpy as np

_tok_probe = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = raw_dataset["train"].shuffle(seed=42).select(range(min(1000, len(raw_dataset["train"]))))
lengths = []
for ex in sample:
    full_text = f"{ex['context']}\n\n{ex['question']}\n\n{ex['answer']}"
    lengths.append(len(_tok_probe(full_text)["input_ids"]))

lengths = np.array(lengths)
print(f"mean={lengths.mean():.0f}  median={np.median(lengths):.0f}  "
      f"p90={np.percentile(lengths, 90):.0f}  p95={np.percentile(lengths, 95):.0f}  "
      f"p99={np.percentile(lengths, 99):.0f}  max={lengths.max()}")
print()
print("If p95 is well under MAX_SEQ_LENGTH, you're safe. If not, either raise "
      "MAX_SEQ_LENGTH (costs VRAM) or accept truncation on the longest tail examples.")

mean=391  median=371  p90=524  p95=567  p99=666  max=863

If p95 is well under MAX_SEQ_LENGTH, you're safe. If not, either raise MAX_SEQ_LENGTH (costs VRAM) or accept truncation on the longest tail examples.


## 4. Format for RAG-style instruction tuning

Each example becomes a chat-formatted sequence: a system prompt establishing the assistant's role, a user turn with the regulatory context + question, and an assistant turn with the answer. We use Qwen3's chat template with `enable_thinking=False` since this is direct QA, not a reasoning task — we don't want the model generating `<think>` blocks for straightforward regulatory lookups.

In [31]:
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant on Reserve Bank of India (RBI) banking regulations. "
    "Answer the question using only the information in the provided regulatory context. "
    "Be precise and comprehensive, and note when the context does not fully answer the question."
)

def build_messages(example):
    user_content = f"Context:\n{example['context']}\n\nQuestion: {example['question']}"
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

train_ds = raw_dataset["train"].map(build_messages, remove_columns=raw_dataset["train"].column_names)
eval_ds = raw_dataset["validation"].map(build_messages, remove_columns=raw_dataset["validation"].column_names)

if TRAIN_SUBSET_SIZE is not None:
    train_ds = train_ds.shuffle(seed=42).select(range(min(TRAIN_SUBSET_SIZE, len(train_ds))))
if EVAL_SUBSET_SIZE is not None:
    eval_ds = eval_ds.shuffle(seed=42).select(range(min(EVAL_SUBSET_SIZE, len(eval_ds))))

print(f"train: {len(train_ds)}  |  eval: {len(eval_ds)}")
print()
print("Example formatted record:")
print(train_ds[0])

train: 3000  |  eval: 200

Example formatted record:
{'messages': [{'role': 'system', 'content': 'You are a knowledgeable assistant on Reserve Bank of India (RBI) banking regulations. Answer the question using only the information in the provided regulatory context. Be precise and comprehensive, and note when the context does not fully answer the question.'}, {'role': 'user', 'content': "Context:\nMaster Direction on Outsourcing of Information Technology Services Regulated Entities (REs) have been extensively leveraging Information Technology (IT) and IT enabled Services (ITeS) to support their business models, products and services offered to their customers. REs also outsource substantial portion of their IT activities to third parties, which expose them to various risks. In order to ensure effective management of attendant risks, the Statement on Developmental and Regulatory Policies dated February 10, 2022, proposed the issuance of suitable regulatory guidelines on Outsourcing of I

## 5. Load model in 4-bit (QLoRA)

Loads `Qwen/Qwen3.5-4B` directly -- no fallback model here (this notebook is dedicated to this
one model). `AutoModelForCausalLM` resolves the `qwen3_5` model type to the text-only
`Qwen3_5ForCausalLM` class and skips loading the vision-tower weights automatically (verified
in the cell below by checking that no `visual`/`vision` keys ended up in the loaded model and
that the model's class name is the one we expect) -- so even though the checkpoint includes
multimodal weights, what actually lands on your GPU is a text-only decoder.


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

_compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=_compute_dtype,
    bnb_4bit_use_double_quant=True,
)

def load_model_and_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=_compute_dtype,
    )
    model.config.use_cache = False  # required for gradient checkpointing; re-enabled before generation later
    return model, tok

# No try/except fallback here -- this notebook is dedicated to Qwen3.5-4B. If you OOM even at
# 4-bit on a T4 (15GB), that would be surprising given Qwen3-4B fits comfortably in the same
# setup; if it happens, restart the runtime and try dropping MAX_SEQ_LENGTH in the config cell.
active_model_name = MODEL_NAME
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
print(f"Loaded {MODEL_NAME} successfully.")
print("Active model:", active_model_name)
print("GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))

# --- Defensive check: confirm this actually loaded as text-only, not the full multimodal model ---
# Qwen3_5ForCausalLM should have no vision-tower submodule. If this assertion ever fails, it
# means a transformers version change resolved AutoModelForCausalLM differently than expected --
# better to catch that here than discover it three cells later as a confusing shape mismatch.
_model_class_name = type(model).__name__
_has_vision_weights = any("visual" in name or "vision" in name for name, _ in model.named_parameters())
print(f"\nLoaded model class: {_model_class_name}")
print(f"Vision-tower weights present: {_has_vision_weights} (expected: False)")
assert not _has_vision_weights, (
    "Vision weights were loaded -- this should not happen with AutoModelForCausalLM on a "
    "qwen3_5 checkpoint. Check your transformers version (needs >= 5.2.0)."
)
print("Confirmed text-only load, as expected.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded Qwen/Qwen3.5-4B successfully.
Active model: Qwen/Qwen3.5-4B
GPU memory allocated: 3.12 GB

Loaded model class: Qwen3_5ForCausalLM
Vision-tower weights present: False (expected: False)
Confirmed text-only load, as expected.


## 6. Prepare for k-bit training and define the LoRA config

**Important**: we deliberately do NOT call `get_peft_model()` here. `SFTTrainer` applies its memory-efficient
chunked cross-entropy patch (`loss_type="chunked_nll"`, the default) by patching `.forward()` on the inner
causal LM. For that patch to land correctly on a LoRA-wrapped model, TRL needs to do the PEFT-wrapping itself
(via `peft_config=` below) — wrapping the model ourselves first and handing `SFTTrainer` an already-PeftModel
instance can cause the chunked patch to silently miss, which is what produced the earlier
`entropy_from_logits` OOM: the model fell back to computing full, unchunked `[batch*seq, vocab]` logits.

In [32]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=USE_GRAD_CHECKPOINTING)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,  # do NOT include "lm_head" here — incompatible with loss_type="chunked_nll"
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA config ready (not yet applied). SFTTrainer will wrap the model with this in the next step.")

LoRA config ready (not yet applied). SFTTrainer will wrap the model with this in the next step.


## 7. Tokenize with chat template (thinking mode OFF)

Qwen3.5 thinks by default (more aggressively than Qwen3 -- it's described as the model's primary mode, not just an option), but the same `enable_thinking=False` argument to `apply_chat_template` disables it, confirmed against Qwen3.5 usage examples. This is RAG QA, not multi-step reasoning, so non-thinking mode is the right fit here, same as Qwen3.

In [33]:
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return text

# Sanity check on one example
print(formatting_func(train_ds[0])[:1500])

<|im_start|>system
You are a knowledgeable assistant on Reserve Bank of India (RBI) banking regulations. Answer the question using only the information in the provided regulatory context. Be precise and comprehensive, and note when the context does not fully answer the question.<|im_end|>
<|im_start|>user
Context:
Master Direction on Outsourcing of Information Technology Services Regulated Entities (REs) have been extensively leveraging Information Technology (IT) and IT enabled Services (ITeS) to support their business models, products and services offered to their customers. REs also outsource substantial portion of their IT activities to third parties, which expose them to various risks. In order to ensure effective management of attendant risks, the Statement on Developmental and Regulatory Policies dated February 10, 2022, proposed the issuance of suitable regulatory guidelines on Outsourcing of IT Services. Accordingly, a draft Master Direction on Outsourcing of IT Services was r

## 8. Train with TRL's SFTTrainer

`paged_adamw_8bit` is the key setting that makes this fit on small VRAM — it streams optimizer states to CPU when GPU memory is tight, at a small speed cost. Don't swap this for plain `adamw_torch` on a 6GB card.

**Note**: newer TRL versions renamed `SFTConfig`'s `max_seq_length` argument to `max_length` (the old name was deprecated, then removed). This notebook uses `max_length`. If you're on an older pinned TRL (<0.20) and get an unrecognized-argument error in the opposite direction, swap it back to `max_seq_length`. Run `import trl; print(trl.__version__)` if you want to check which one applies.

In [34]:
from trl import SFTTrainer, SFTConfig
import os

# WANDB_PROJECT is read automatically by transformers/accelerate from this environment variable —
# it's how the project name set in the config cell actually reaches W&B's servers.
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,                              # checkpoints/logs saved here

    # --- Batch sizing (see config cell for full explanation of micro-batch vs effective batch) ---
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # --- Core training hyperparameters ---
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    # cosine schedule: the learning rate follows a smooth downward curve (shaped like one hump
    # of a cosine wave) from its peak down to ~0 by the end of training, rather than staying flat
    # the whole time. This tends to produce slightly better final results than a constant rate,
    # since the model takes smaller, more careful steps as it gets closer to convergence.
    lr_scheduler_type="cosine",

    # --- How often to log / checkpoint / evaluate (see config cell for exact meanings) ---
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",     # evaluate every EVAL_STEPS steps (not just once per epoch)
    save_strategy="steps",     # checkpoint every SAVE_STEPS steps
    save_total_limit=2,        # only keep the 2 most recent checkpoints on disk (saves space)

    # --- Memory-saving settings (important for fitting on limited VRAM) ---
    # paged_adamw_8bit: a memory-efficient version of the Adam optimizer. Normal Adam keeps two
    # extra numbers (momentum + variance) PER PARAMETER, which doubles memory usage on top of the
    # parameters themselves. This variant stores those extra numbers in 8-bit instead of 32-bit,
    # AND can temporarily move them to CPU RAM ("paging") if GPU memory gets tight — similar to
    # how an OS pages memory to disk under pressure. This is the single most important setting
    # for fitting training on a small GPU.
    optim="paged_adamw_8bit",

    # bf16/fp16: which reduced-precision number format to train in (instead of full 32-bit floats,
    # which would use 2x the memory). USE_BF16 is decided automatically earlier in the notebook —
    # currently always True, because Qwen3's own weights are natively bf16 and training in fp16
    # instead causes a crash (see the GPU-detection cell's comments for the full explanation).
    bf16=USE_BF16,
    fp16=not USE_BF16,

    # gradient_checkpointing: normally, during the forward pass, the model keeps every
    # intermediate calculation in memory so it can use them later during backpropagation. Instead,
    # this setting throws most of them away and RECOMPUTES them when needed — trading extra
    # computation time for a large reduction in memory usage. USE_GRAD_CHECKPOINTING is set by the
    # GPU-detection cell (currently always on, as a safety margin).
    gradient_checkpointing=USE_GRAD_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False} if USE_GRAD_CHECKPOINTING else None,

    # max_length: the sequence-length cap explained in detail in the config cell above.
    max_length=MAX_SEQ_LENGTH,

    # assistant_only_loss: ensures the model is only "graded" (loss computed) on the tokens it
    # itself needs to generate -- the answer -- and NOT on the system prompt or the question.
    #
    # IMPORTANT FOR QWEN3.5: this only works correctly because we explicitly pass
    # processing_class=tokenizer to SFTTrainer below. Without that, SFTTrainer calls
    # AutoProcessor.from_pretrained(model_id) to auto-detect a processing class -- and because
    # the Qwen/Qwen3.5-4B Hub repo is the full multimodal checkpoint, AutoProcessor resolves to
    # a multimodal ProcessorMixin REGARDLESS of which model class we actually instantiated
    # (Qwen3_5ForCausalLM, text-only). SFTTrainer treats getting back a ProcessorMixin as proof
    # the model is a VLM (self._is_vlm = True) and behaves accordingly from then on: it blocks
    # assistant_only_loss=True outright, and separately, its chunked-loss patch reads
    # model.config.text_config (the nested-config layout multimodal wrappers use) instead of
    # model.config directly -- which crashes with "'Qwen3_5TextConfig' object has no attribute
    # 'text_config'" because our model's config IS the text config already, not a wrapper around
    # one. Passing processing_class=tokenizer explicitly skips the AutoProcessor auto-detection,
    # so SFTTrainer correctly sets self._is_vlm = False and both problems go away.
    assistant_only_loss=True,

    # --- Experiment tracking: send live metrics to BOTH TensorBoard (local, no account needed)
    # AND Weights & Biases (cloud dashboard, needs the login from the cell near the top). Having
    # both means you can watch live in-notebook (TensorBoard) AND keep a permanent, shareable,
    # comparison-friendly record across runs (W&B) without extra effort.
    report_to=["tensorboard", "wandb"],
    logging_dir=f"{OUTPUT_DIR}/tb_logs",   # where TensorBoard's local log files go
    logging_strategy="steps",
    run_name=RUN_NAME,                      # the human-readable run label set in the config cell

    # load_best_model_at_end + metric_for_best_model: at the very end of training, instead of
    # keeping whichever checkpoint happened to be saved last, reload whichever checkpoint had the
    # LOWEST validation loss seen at any point during training. This protects against the case
    # where the model started overfitting near the end and its final state is actually worse than
    # an earlier one.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # loss_type="chunked_nll": computes the cross-entropy loss in small chunks instead of all at
    # once. Qwen3.5's vocabulary is ~248,000 possible tokens (about 1.6x Qwen3's ~152,000) --
    # computing the loss the "normal" way would require building a
    # [batch_size x sequence_length, 248000] tensor in one shot, which is even larger here than
    # for the Qwen3 notebooks and was the direct cause of an out-of-memory crash in an earlier
    # version of the Qwen3 notebook this one is based on. Chunking keeps memory usage flat
    # regardless of sequence length or vocab size.
    loss_type="chunked_nll",
)

import json
from transformers import TrainerCallback

class JSONLLoggerCallback(TrainerCallback):
    """A third logging destination, alongside TensorBoard and W&B: a plain local text file.

    Every time the trainer logs a metric (loss, eval_loss, learning_rate, etc. — the same
    numbers going to TensorBoard/W&B), this callback also appends it as one line of JSON to
    `training_log.jsonl`. This is useful because it's a plain file you can open, grep, or load
    with pandas at any time, completely independent of whether TensorBoard or W&B are working —
    a good fallback if either of those has a hiccup, and an easy way to keep a permanent copy
    since the Colab VM itself gets wiped on disconnect (W&B's cloud copy survives that too, but
    having a local file as well costs nothing).
    """
    def __init__(self, log_path):
        self.log_path = log_path
        open(self.log_path, "w").close()  # create/empty the file fresh at the start of this run

    def on_log(self, args, state, control, logs=None, **kwargs):
        # `on_log` is automatically called by the Trainer every time it has new metrics to report
        # (i.e. every LOGGING_STEPS steps, and at each evaluation). `logs` is a dict like
        # {"loss": 1.83, "learning_rate": 0.00019, ...} or {"eval_loss": 1.71} during evaluation.
        if logs is None:
            return
        record = dict(logs)
        record["step"] = state.global_step   # which training step this metric was recorded at
        record["epoch"] = state.epoch         # how many full passes over the data, as a fraction
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

TRAINING_LOG_PATH = f"{OUTPUT_DIR}/training_log.jsonl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=formatting_func,
    # processing_class=tokenizer: explicitly pass our already-loaded tokenizer instead of
    # letting SFTTrainer auto-detect a processing class via AutoProcessor.from_pretrained(). On
    # Qwen3.5 specifically, that auto-detection resolves to a multimodal processor (because the
    # Hub repo is the full multimodal checkpoint) even though we loaded the text-only model
    # class -- which makes SFTTrainer misclassify this as a VLM and breaks both
    # assistant_only_loss and the chunked-loss patch. See the comment on assistant_only_loss
    # above for the full explanation. Passing the tokenizer directly sidesteps this entirely.
    processing_class=tokenizer,
    # peft_config (not a manually pre-wrapped model!): SFTTrainer applies the LoRA wrapping
    # itself when given the config this way, which is required for its internal memory-saving
    # patches (like chunked_nll above) to correctly recognize the model's structure. Wrapping the
    # model ourselves first caused a silent failure earlier in this notebook's development.
    peft_config=lora_config,
    callbacks=[JSONLLoggerCallback(TRAINING_LOG_PATH)],
)

# Sanity check: confirms LoRA is actually active and shows what fraction of the model's
# parameters are trainable. Expect well under 1% (typically 0.1-0.5%) — if this prints something
# close to 100%, LoRA did not get applied correctly and you'd effectively be full-fine-tuning,
# which would be far too slow/memory-hungry for this setup.
trainer.model.print_trainable_parameters()
print(f"W&B project: {WANDB_PROJECT}  |  Run name: {RUN_NAME}")
print(f"Training metrics will also be appended locally to: {TRAINING_LOG_PATH}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

trainable params: 1,572,864 || all params: 4,207,324,160 || trainable%: 0.0374
W&B project: qwen3-rbi-qa-finetune  |  Run name: Qwen3.5-4B_subset3000_ep1_20260626-080953
Training metrics will also be appended locally to: ./qwen3.5-4b-rbi-qa-lora/training_log.jsonl


### (Optional) Launch TensorBoard inline

Run this before or during training to watch loss curves live inside the notebook. It reads from
`logging_dir` (`{OUTPUT_DIR}/tb_logs`), which `SFTConfig`'s `report_to="tensorboard"` writes to automatically
— no extra account or API key needed, unlike Weights & Biases.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tb_logs

### Diagnostic: confirm which precision mode is actually active

Run this once before `trainer.train()`. It checks every layer where an fp16 vs bf16 mismatch could be
hiding: the `ACCELERATE_MIXED_PRECISION` env var (which `TrainingArguments`/`SFTConfig` sets as a process-wide
side effect and which can persist oddly across runs in the same kernel), the resolved `accelerate` state, our
own config flags, and — most importantly — the actual dtype of every distinct parameter/buffer group in the
live model, since PEFT or `bitsandbytes` could be creating something in fp16 regardless of what we asked for.

In [35]:
import os
from collections import Counter

print("--- Env var ---")
print("ACCELERATE_MIXED_PRECISION =", os.environ.get("ACCELERATE_MIXED_PRECISION", "<not set>"))

print("\n--- Our config flags ---")
print("USE_BF16 =", USE_BF16)
print("sft_config.bf16 =", sft_config.bf16, " sft_config.fp16 =", sft_config.fp16)

print("\n--- Live accelerate state (the actual source of truth for GradScaler) ---")
try:
    accel_state = trainer.accelerator.state
    print("accelerator.mixed_precision =", trainer.accelerator.mixed_precision)
    print("accelerator.scaler =", trainer.accelerator.scaler)
except Exception as e:
    print("Could not read trainer.accelerator state:", e)

print("\n--- Actual parameter/buffer dtypes in the live model (grouped by dtype) ---")
dtype_counts = Counter()
dtype_examples = {}
for name, p in trainer.model.named_parameters():
    dtype_counts[p.dtype] += 1
    if p.dtype not in dtype_examples:
        dtype_examples[p.dtype] = name
for dtype, count in dtype_counts.items():
    print(f"{dtype}: {count} params  (example: {dtype_examples[dtype]})")

print("\nIf you see torch.float16 anywhere above alongside bf16, or if "
      "ACCELERATE_MIXED_PRECISION says 'fp16', paste this whole output back.")

--- Env var ---
ACCELERATE_MIXED_PRECISION = <not set>

--- Our config flags ---
USE_BF16 = True
sft_config.bf16 = True  sft_config.fp16 = False

--- Live accelerate state (the actual source of truth for GradScaler) ---
accelerator.mixed_precision = bf16
accelerator.scaler = None

--- Actual parameter/buffer dtypes in the live model (grouped by dtype) ---
torch.float32: 178 params  (example: base_model.model.model.embed_tokens.weight)
torch.uint8: 248 params  (example: base_model.model.model.layers.0.linear_attn.out_proj.weight)
torch.bfloat16: 64 params  (example: base_model.model.model.layers.3.self_attn.q_proj.lora_A.default.weight)

If you see torch.float16 anywhere above alongside bf16, or if ACCELERATE_MIXED_PRECISION says 'fp16', paste this whole output back.


### Live runtime check against the 5.5h budget (read this one -- the estimate here is the shakiest of the three notebooks)

Runs a handful of real forward+backward steps through `trainer`'s own dataloader and optimizer
-- the same code path `trainer.train()` will use -- to measure actual seconds/step on *your*
assigned T4, without permanently consuming any training steps or touching the LR
scheduler/W&B/TensorBoard logging. Compare the result against the ~96 sec/step estimate from the
runtime-budget cell near the top. Given the real architectural unknowns for this model (hybrid
linear/full attention, larger vocab, no guaranteed fast DeltaNet kernels), this measurement
matters more here than in the other two notebooks in this series -- if your real number comes in
notably higher than 96 sec/step, lower `TRAIN_SUBSET_SIZE` in the config cell and re-run from
there before committing to the long run in the next cell.


In [38]:
import time, math, torch

_PROBE_STEPS = 5
_probe_loader = trainer.get_train_dataloader()
_probe_iter = iter(_probe_loader)

trainer.model.train()
_t0 = time.time()
for _ in range(_PROBE_STEPS):
    _batch = next(_probe_iter)
    _batch = {k: v.to(trainer.model.device) if hasattr(v, "to") else v for k, v in _batch.items()}
    _loss = trainer.compute_loss(trainer.model, _batch)
    if isinstance(_loss, tuple):
        _loss = _loss[0]
        _loss.backward()
    trainer.model.zero_grad(set_to_none=True)  # discard probe gradients -- this run doesn't count
torch.cuda.synchronize()
_elapsed = time.time() - _t0
_measured_sec_per_step = _elapsed / _PROBE_STEPS

_steps_per_epoch = math.ceil(len(train_ds) / (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
_total_steps = _steps_per_epoch * NUM_EPOCHS
_projected_total_hours = (_measured_sec_per_step * _total_steps) / 3600

print(f"Measured: {_measured_sec_per_step:.1f} sec/step over a {_PROBE_STEPS}-step probe on this GPU")
print(f"(Upfront estimate was ~96 sec/step -- see the runtime-budget cell for how that number was derived)")
print(f"Planned total steps for the real run: {_total_steps}")
print(f"Projected training time at this speed: {_projected_total_hours:.2f}h "
      f"(plus ~12 min already spent on installs/downloads)")
if _projected_total_hours > 4.5:
    print()
    print("WARNING: projected time is close to or over the 5.5h budget. Consider lowering "
          "TRAIN_SUBSET_SIZE in the config cell and re-running from there before continuing.")
else:
    print()
    print("Within budget -- safe to proceed to trainer.train() in the next cell.")

torch.cuda.empty_cache()


Measured: 2.0 sec/step over a 5-step probe on this GPU
(Upfront estimate was ~96 sec/step -- see the runtime-budget cell for how that number was derived)
Planned total steps for the real run: 188
Projected training time at this speed: 0.11h (plus ~12 min already spent on installs/downloads)

Within budget -- safe to proceed to trainer.train() in the next cell.


In [39]:
import torch
torch.cuda.empty_cache()
print("Starting the full training run (the probe above doesn't count toward this -- gradients "
      "from the probe were discarded and the LR scheduler/optimizer haven't taken any real steps).")
trainer.train()

Starting the full training run (the probe above doesn't count toward this -- gradients from the probe were discarded and the LR scheduler/optimizer haven't taken any real steps).


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 9. Save the LoRA adapter

In [ ]:
FINAL_ADAPTER_DIR = OUTPUT_DIR + "-final"
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print("Saved adapter + tokenizer to", FINAL_ADAPTER_DIR)
print("Trained on:", active_model_name)

# Explicitly close out the W&B run now that training + saving are done. Without this, the run
# can be left showing as "still running" on your wandb.ai dashboard until it eventually times
# out on its own — calling finish() marks it complete immediately and uploads any remaining
# buffered data.
import wandb
wandb.finish()

## 10. Quick qualitative test

In [ ]:
# Use trainer.model — the actual LoRA-wrapped, trained model. The bare `model` variable
# from Cell 5 is the unwrapped base model and was never trained (SFTTrainer applied the
# LoRA wrapping internally via peft_config=).
gen_model = trainer.model
gen_model.eval()
gen_model.config.use_cache = True  # was disabled for training (gradient checkpointing); re-enable for fast generation

test_example = eval_ds[0]["messages"][:2]  # system + user only, drop the gold answer
prompt_text = tokenizer.apply_chat_template(
    test_example, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(prompt_text, return_tensors="pt").to(gen_model.device)

with torch.no_grad():
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("QUESTION CONTEXT:\n", test_example[1]["content"][:500], "...\n")
print("MODEL ANSWER:\n", generated, "\n")
print("GOLD ANSWER:\n", eval_ds[0]["messages"][2]["content"])

## 11. Review training logs

Reads back `training_log.jsonl` (written live by `JSONLLoggerCallback` during training) and plots train/eval
loss. This works even if the TensorBoard cell above was never opened, and gives you a plain file you can
download from Colab and keep alongside the model — useful since the Colab VM itself is wiped on disconnect.

In [ ]:
import json
import matplotlib.pyplot as plt

records = []
with open(TRAINING_LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} logged events from {TRAINING_LOG_PATH}")
print()

train_steps = [r["step"] for r in records if "loss" in r]
train_loss = [r["loss"] for r in records if "loss" in r]
eval_steps = [r["step"] for r in records if "eval_loss" in r]
eval_loss = [r["eval_loss"] for r in records if "eval_loss" in r]

if train_loss:
    plt.figure(figsize=(8, 4))
    plt.plot(train_steps, train_loss, label="train loss", marker="o", markersize=3)
    if eval_loss:
        plt.plot(eval_steps, eval_loss, label="eval loss", marker="s", markersize=4)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(f"Training run: {active_model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No training-loss events found yet — run trainer.train() first, or check "
          "LOGGING_STEPS isn't larger than your total step count.")

## Notes for moving this to your local RTX 3050

1. **Drop `TRAIN_SUBSET_SIZE` to `None` only after** you've confirmed the pipeline runs clean here in the cloud — the full 19,113-example train set will take meaningfully longer on a 3050 than on a cloud T4, and the per-step timing uncertainty for this architecture (see the runtime-budget cell near the top) makes an upfront full-dataset commitment riskier here than for the other two notebooks.
2. **On the 3050 specifically**: Qwen3.5-4B's larger vocabulary (~248K vs Qwen3's ~152K) means a somewhat bigger embedding table and LM head than Qwen3-4B at the same hyperparameters -- if you OOM where the 4B Qwen3 notebook didn't, that extra vocab overhead is the most likely first culprit. Try, in order: (a) drop `MAX_SEQ_LENGTH` to 256, (b) drop `LORA_TARGET_MODULES` to just `["q_proj", "v_proj"]`, (c) switch to `Qwen/Qwen3.5-2B` or `Qwen/Qwen3.5-0.8B` if available for your use case.
3. **Close other GPU-using applications** (browser hardware acceleration, other notebooks) before running locally — on a 6GB card, 500MB of background usage is the difference between fitting and OOMing.
4. Windows users: bitsandbytes 4-bit support on Windows has historically lagged Linux; if you hit import errors, run this under WSL2 rather than native Windows Python.
5. This notebook checkpoints every `SAVE_STEPS` steps — if your local training session gets interrupted, you can resume with `trainer.train(resume_from_checkpoint=True)`.
6. If you have the `causal_conv1d` and `fla` packages installed locally (the optional fast kernels for the Gated DeltaNet layers -- not installed by default on Colab, and not assumed by this notebook's runtime budget), Qwen3.5-4B may train noticeably faster locally than the Colab estimate suggests. Worth installing if you're doing repeated local runs.

In [ ]:
!zip -r qwen3.5-4b.zip /content/qwen3.5-4b-rbi-qa-lora-final